In [ ]:
# This is a Demo! Not for medical use!

import pandas as pd
import transformers
import torch
print(transformers.__version__)
print(torch.__version__)
from transformers import T5Tokenizer, T5ForConditionalGeneration
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

# Erweiterte 100-Beispiele Datenbasis für besseres Training

data = [
    # Pneumologie
    {"input": "Symptome: Fieber, Husten. CRP: 67. Bildgebung: Infiltrat basal rechts. Was ist die wahrscheinlichste Diagnose?", "output": "Pneumonie"},
    {"input": "Symptome: Dyspnoe, Beinschwellung links. D-Dimer erhöht. Was ist die wahrscheinlichste Diagnose?", "output": "Lungenembolie"},
    {"input": "Symptome: Chronischer Husten, Gewichtsverlust, Nachtschweiß. Was ist die wahrscheinlichste Diagnose?", "output": "Tuberkulose"},
    {"input": "Symptome: Akute Dyspnoe, Thoraxschmerz, Pneumothorax im Röntgen. Was ist die wahrscheinlichste Diagnose?", "output": "Pneumothorax"},
    {"input": "Symptome: Chronische Dyspnoe, Raucheranamnese, FEV1 reduziert. Was ist die wahrscheinlichste Diagnose?", "output": "COPD"},
    
    # Kardiologie
    {"input": "Symptome: Müdigkeit, Blässe. Hb: niedrig. Was ist die wahrscheinlichste Diagnose?", "output": "Anämie"},
    {"input": "Symptome: Brustschmerz, Troponin hoch, EKG ST-Hebung. Was ist die wahrscheinlichste Diagnose?", "output": "Herzinfarkt"},
    {"input": "Symptome: Belastungsdyspnoe, Beinödeme, BNP erhöht. Was ist die wahrscheinlichste Diagnose?", "output": "Herzinsuffizienz"},
    {"input": "Symptome: Palpitationen, Herzrasen, RR 180/110. Was ist die wahrscheinlichste Diagnose?", "output": "Hypertensive Krise"},
    {"input": "Symptome: Synkope, Systolikum, Echokardiographie: Aortenstenose. Was ist die wahrscheinlichste Diagnose?", "output": "Aortenstenose"},
    
    # Endokrinologie
    {"input": "Symptome: Polyurie, Polydipsie, BZ 320 mg/dl. Was ist die wahrscheinlichste Diagnose?", "output": "Diabetes mellitus"},
    {"input": "Symptome: Gewichtsverlust, Tachykardie, TSH supprimiert, fT4 erhöht. Was ist die wahrscheinlichste Diagnose?", "output": "Hyperthyreose"},
    {"input": "Symptome: Gewichtszunahme, Müdigkeit, TSH erhöht, fT4 niedrig. Was ist die wahrscheinlichste Diagnose?", "output": "Hypothyreose"},
    {"input": "Symptome: Hypoglykämie, Schwitzen, Insulin erhöht. Was ist die wahrscheinlichste Diagnose?", "output": "Insulinom"},
    {"input": "Symptome: Hyponatriämie, Hyperkaliämie, Hypotonie. Was ist die wahrscheinlichste Diagnose?", "output": "Morbus Addison"},
    
    # Gastroenterologie
    {"input": "Symptome: Epigastrische Schmerzen, Übelkeit, Lipase erhöht. Was ist die wahrscheinlichste Diagnose?", "output": "Pankreatitis"},
    {"input": "Symptome: Ikterus, Oberbauchschmerz, erhöhte Leberwerte. Was ist die wahrscheinlichste Diagnose?", "output": "Hepatitis"},
    {"input": "Symptome: Durchfall, Gewichtsverlust, CRP erhöht. Was ist die wahrscheinlichste Diagnose?", "output": "Morbus Crohn"},
    {"input": "Symptome: Teerstuhl, Hb-Abfall, Magenschmerzen. Was ist die wahrscheinlichste Diagnose?", "output": "Ulkusblutung"},
    {"input": "Symptome: Aszites, Splenomegalie, Bilirubin erhöht. Was ist die wahrscheinlichste Diagnose?", "output": "Leberzirrhose"},
    
    # Neurologie
    {"input": "Symptome: Hemiparese links, Aphasie, CT: Hypodensität rechts. Was ist die wahrscheinlichste Diagnose?", "output": "Schlaganfall"},
    {"input": "Symptome: Kopfschmerzen, Lichtscheu, Meningismus, Liquor trüb. Was ist die wahrscheinlichste Diagnose?", "output": "Meningitis"},
    {"input": "Symptome: Tremor, Rigor, Bradykinesie. Was ist die wahrscheinlichste Diagnose?", "output": "Morbus Parkinson"},
    {"input": "Symptome: Sehstörungen, Parästhesien, MRT: Multiple Läsionen. Was ist die wahrscheinlichste Diagnose?", "output": "Multiple Sklerose"},
    {"input": "Symptome: Generalisierter Krampfanfall, EEG: Spike-Wave-Komplexe. Was ist die wahrscheinlichste Diagnose?", "output": "Epilepsie"},
    
    # Nephrologie
    {"input": "Symptome: Ödeme, Proteinurie, Hypoalbuminämie. Was ist die wahrscheinlichste Diagnose?", "output": "Nephrotisches Syndrom"},
    {"input": "Symptome: Oligurie, Kreatinin erhöht, Harnstoff erhöht. Was ist die wahrscheinlichste Diagnose?", "output": "Niereninsuffizienz"},
    {"input": "Symptome: Flankenschmerz, Hämaturie, Sonographie: Nephrolithiasis. Was ist die wahrscheinlichste Diagnose?", "output": "Nierensteine"},
    {"input": "Symptome: Dysurie, Pollakisurie, Urin: Leukozyten, Bakterien. Was ist die wahrscheinlichste Diagnose?", "output": "Harnwegsinfekt"},
    {"input": "Symptome: Hypertonie, Kreatinin chronisch erhöht, Anämie. Was ist die wahrscheinlichste Diagnose?", "output": "Chronische Niereninsuffizienz"},
    
    # Hämatologie/Onkologie
    {"input": "Symptome: Fatigue, Petechien, Thrombozyten niedrig. Was ist die wahrscheinlichste Diagnose?", "output": "Thrombozytopenie"},
    {"input": "Symptome: Lymphknotenschwellung, B-Symptome, Biopsie: Reed-Sternberg-Zellen. Was ist die wahrscheinlichste Diagnose?", "output": "Morbus Hodgkin"},
    {"input": "Symptome: Blasten im Blutbild, Anämie, Thrombozytopenie. Was ist die wahrscheinlichste Diagnose?", "output": "Leukämie"},
    {"input": "Symptome: Knochenschmerzen, Hyperkalzämie, M-Gradient. Was ist die wahrscheinlichste Diagnose?", "output": "Multiples Myelom"},
    {"input": "Symptome: Splenomegalie, Leukozytose, Philadelphia-Chromosom. Was ist die wahrscheinlichste Diagnose?", "output": "Chronische myeloische Leukämie"},
    
    # Infektiologie
    {"input": "Symptome: Fieber, Schüttelfrost, Blutkulturen: Staphylococcus aureus. Was ist die wahrscheinlichste Diagnose?", "output": "Sepsis"},
    {"input": "Symptome: Durchfall, Fieber, Stuhlkultur: Salmonellen. Was ist die wahrscheinlichste Diagnose?", "output": "Salmonellose"},
    {"input": "Symptome: Husten, Fieber, Atypische Pneumonie, Mycoplasma-Titer erhöht. Was ist die wahrscheinlichste Diagnose?", "output": "Mycoplasma-Pneumonie"},
    {"input": "Symptome: Exanthem, Fieber, Lymphadenopathie, EBV-Serologie positiv. Was ist die wahrscheinlichste Diagnose?", "output": "EBV-Infektion"},
    {"input": "Symptome: Opportunistische Infekte, CD4 < 200, HIV-Test positiv. Was ist die wahrscheinlichste Diagnose?", "output": "AIDS"},
    
    # Rheumatologie
    {"input": "Symptome: Morgensteifigkeit, Polyarthritis, RF positiv, Anti-CCP positiv. Was ist die wahrscheinlichste Diagnose?", "output": "Rheumatoide Arthritis"},
    {"input": "Symptome: Raynaud-Phänomen, Sklerodaktylie, ANA positiv. Was ist die wahrscheinlichste Diagnose?", "output": "Systemische Sklerose"},
    {"input": "Symptome: Schmetterlingserythem, Arthritis, ANA positiv, Anti-dsDNA positiv. Was ist die wahrscheinlichste Diagnose?", "output": "Lupus erythematodes"},
    {"input": "Symptome: Großzehengrundgelenk geschwollen, Harnsäure erhöht. Was ist die wahrscheinlichste Diagnose?", "output": "Gicht"},
    {"input": "Symptome: Rückenschmerzen, HLA-B27 positiv, Sakroiliitis im MRT. Was ist die wahrscheinlichste Diagnose?", "output": "Morbus Bechterew"},
    
    # Dermatologie
    {"input": "Symptome: Juckende Plaques, Schuppung, Auspitz-Phänomen. Was ist die wahrscheinlichste Diagnose?", "output": "Psoriasis"},
    {"input": "Symptome: Asymmetrisches Pigmentmal, ABCDE-Kriterien auffällig. Was ist die wahrscheinlichste Diagnose?", "output": "Malignes Melanom"},
    {"input": "Symptome: Erysipelartige Rötung, Überwärmung, CRP erhöht. Was ist die wahrscheinlichste Diagnose?", "output": "Erysipel"},
    {"input": "Symptome: Vesikelbläschen im Dermatom, brennende Schmerzen. Was ist die wahrscheinlichste Diagnose?", "output": "Herpes zoster"},
    {"input": "Symptome: Nässen, Juckreiz, Kontakt mit Allergen. Was ist die wahrscheinlichste Diagnose?", "output": "Kontaktekzem"},
    
    # Orthopädie
    {"input": "Symptome: Rückenschmerzen, Ischialgie, MRT: Bandscheibenvorfall L5/S1. Was ist die wahrscheinlichste Diagnose?", "output": "Bandscheibenvorfall"},
    {"input": "Symptome: Knieschmerzen, Bewegungseinschränkung, Arthrose im Röntgen. Was ist die wahrscheinlichste Diagnose?", "output": "Gonarthrose"},
    {"input": "Symptome: Nackenschmerzen nach Trauma, HWS-Röntgen: Fraktur C6. Was ist die wahrscheinlichste Diagnose?", "output": "HWS-Fraktur"},
    {"input": "Symptome: Schulterschmerzen, Bewegungseinschränkung, MRT: Ruptur Rotatorenmanschette. Was ist die wahrscheinlichste Diagnose?", "output": "Rotatorenmanschetten-Ruptur"},
    {"input": "Symptome: Handgelenkschmerzen, nächtliche Parästhesien, Tinel-Zeichen positiv. Was ist die wahrscheinlichste Diagnose?", "output": "Karpaltunnelsyndrom"},
    
    # Urologie
    {"input": "Symptome: PSA erhöht, Prostatastanzbiopsie: Adenokarzinom. Was ist die wahrscheinlichste Diagnose?", "output": "Prostatakarzinom"},
    {"input": "Symptome: Pollakisurie, schwacher Harnstrahl, Restharn. Was ist die wahrscheinlichste Diagnose?", "output": "Benigne Prostatahyperplasie"},
    {"input": "Symptome: Testisschwellung, Schmerzen, Sono: inhomogene Struktur. Was ist die wahrscheinlichste Diagnose?", "output": "Hodentumor"},
    {"input": "Symptome: Hämaturie, Gewichtsverlust, CT: Raumforderung Niere. Was ist die wahrscheinlichste Diagnose?", "output": "Nierenzellkarzinom"},
    {"input": "Symptome: Kolikartige Flankenschmerzen, Hämaturie, CT: Konkremente. Was ist die wahrscheinlichste Diagnose?", "output": "Urolithiasis"},
    
    # Gynäkologie
    {"input": "Symptome: Amenorrhö, HCG positiv, Transvaginalsonographie: intrauterine Schwangerschaft. Was ist die wahrscheinlichste Diagnose?", "output": "Schwangerschaft"},
    {"input": "Symptome: Unterbauchschmerzen, HCG positiv, leere Gebärmutter im Ultraschall. Was ist die wahrscheinlichste Diagnose?", "output": "Extrauteringravidität"},
    {"input": "Symptome: Unregelmäßige Blutungen, Mammographie: spiculierte Raumforderung. Was ist die wahrscheinlichste Diagnose?", "output": "Mammakarzinom"},
    {"input": "Symptome: Zyklusstörungen, Hirsutismus, Ultraschall: polyzystische Ovarien. Was ist die wahrscheinlichste Diagnose?", "output": "PCO-Syndrom"},
    {"input": "Symptome: Postmenopausale Blutung, Endometriumbiopsie: Adenokarzinom. Was ist die wahrscheinlichste Diagnose?", "output": "Endometriumkarzinom"},
    
    # Psychiatrie
    {"input": "Symptome: Niedergeschlagenheit, Antriebslosigkeit, Schlafstörungen seit 6 Wochen. Was ist die wahrscheinlichste Diagnose?", "output": "Depression"},
    {"input": "Symptome: Manische Episode, Größenwahn, vermindertes Schlafbedürfnis. Was ist die wahrscheinlichste Diagnose?", "output": "Bipolare Störung"},
    {"input": "Symptome: Halluzinationen, Wahn, Denkstörungen seit 8 Monaten. Was ist die wahrscheinlichste Diagnose?", "output": "Schizophrenie"},
    {"input": "Symptome: Panikattacken, Agoraphobie, Vermeidungsverhalten. Was ist die wahrscheinlichste Diagnose?", "output": "Panikstörung"},
    {"input": "Symptome: Flashbacks, Hypervigilanz nach Trauma vor 3 Monaten. Was ist die wahrscheinlichste Diagnose?", "output": "PTBS"},
    
    # Ophthalmologie
    {"input": "Symptome: Akuter Sehverlust, Augeninnendruck 45 mmHg, Halosehen. Was ist die wahrscheinlichste Diagnose?", "output": "Glaukom"},
    {"input": "Symptome: Schmerzloser Sehverlust, Funduskopie: blasse Papille. Was ist die wahrscheinlichste Diagnose?", "output": "Optikusneuropathie"},
    {"input": "Symptome: Lichtblitze, Rußregen, Gesichtsfelddefekt. Was ist die wahrscheinlichste Diagnose?", "output": "Netzhautablösung"},
    {"input": "Symptome: Progressiver Sehverlust, Funduskopie: Drusen, Pigmentverschiebungen. Was ist die wahrscheinlichste Diagnose?", "output": "Makuladegeneration"},
    {"input": "Symptome: Doppelbilder, Ptosis, Pupillenstarre. Was ist die wahrscheinlichste Diagnose?", "output": "Okulomotoriusparese"},
    
    # HNO
    {"input": "Symptome: Einseitiger Hörverlust, Tinnitus, MRT: Raumforderung Kleinhirnbrückenwinkel. Was ist die wahrscheinlichste Diagnose?", "output": "Akustikusneurinom"},
    {"input": "Symptome: Drehschwindel, Nystagmus, Lagerungsprobe positiv. Was ist die wahrscheinlichste Diagnose?", "output": "Lagerungsschwindel"},
    {"input": "Symptome: Hörverlust, Vertigo, Tinnitus, Endolymphhydrops. Was ist die wahrscheinlichste Diagnose?", "output": "Morbus Menière"},
    {"input": "Symptome: Halsschmerzen, Fieber, Tonsillen vergrößert mit Belägen. Was ist die wahrscheinlichste Diagnose?", "output": "Tonsillitis"},
    {"input": "Symptome: Nasenverstopfung, Riechstörung, CT: Polypen, Pansinusitis. Was ist die wahrscheinlichste Diagnose?", "output": "Rhinosinusitis"},
    
    # Pädiatrie
    {"input": "Symptome: Fieber, Exanthem, Koplik-Flecken, Kind ungeimpft. Was ist die wahrscheinlichste Diagnose?", "output": "Masern"},
    {"input": "Symptome: Bellender Husten, Stridor, Fieber beim Kleinkind. Was ist die wahrscheinlichste Diagnose?", "output": "Pseudokrupp"},
    {"input": "Symptome: Säugling, Trinkschwäche, schlaffe Lähmungen. Was ist die wahrscheinlichste Diagnose?", "output": "Botulismus"},
    {"input": "Symptome: Gedeihstörung, fettige Stühle, Schweißtest positiv. Was ist die wahrscheinlichste Diagnose?", "output": "Mukoviszidose"},
    {"input": "Symptome: Ikterus prolongatus beim Neugeborenen, direktes Bilirubin erhöht. Was ist die wahrscheinlichste Diagnose?", "output": "Gallengangsatresie"},
    
    # Geriatrie
    {"input": "Symptome: Progrediente Demenz, Gedächtnisstörungen, Alzheimer-Biomarker positiv. Was ist die wahrscheinlichste Diagnose?", "output": "Alzheimer-Demenz"},
    {"input": "Symptome: Stürze, orthostatische Hypotonie, Parkinson-Symptome, Demenz. Was ist die wahrscheinlichste Diagnose?", "output": "Lewy-Körper-Demenz"},
    {"input": "Symptome: Akute Verwirrtheit, Sturz, postoperativ nach Hüft-OP. Was ist die wahrscheinlichste Diagnose?", "output": "Delir"},
    {"input": "Symptome: Gangstörung, Harninkontinenz, Demenz, Hydrocephalus. Was ist die wahrscheinlichste Diagnose?", "output": "Normaldruckhydrocephalus"},
    {"input": "Symptome: Schwindel, Stürze, Multimorbidität, Polypharmazie. Was ist die wahrscheinlichste Diagnose?", "output": "Sturzrisiko"},
    
    # Notfallmedizin
    {"input": "Symptome: Bewusstlosigkeit, Blutzucker 25 mg/dl, Schweiß. Was ist die wahrscheinlichste Diagnose?", "output": "Hypoglykämie"},
    {"input": "Symptome: Thoraxschmerz, Schock, D-Dimer stark erhöht, CT-Angiographie: Lungenembolie. Was ist die wahrscheinlichste Diagnose?", "output": "Fulminante Lungenembolie"},
    {"input": "Symptome: Bauchschmerzen, Abwehrspannung, Röntgen: freie Luft. Was ist die wahrscheinlichste Diagnose?", "output": "Perforation"},
    {"input": "Symptome: Starke Kopfschmerzen, Meningismus, CT: Subarachnoidalblutung. Was ist die wahrscheinlichste Diagnose?", "output": "Subarachnoidalblutung"},
    {"input": "Symptome: Reißende Thoraxschmerzen, Blutdruckdifferenz, CT: Aortendissektion. Was ist die wahrscheinlichste Diagnose?", "output": "Aortendissektion"},
    
    # Weitere medizinische Fachbereiche
    {"input": "Symptome: Rötung, Schwellung, Überwärmung, Funktionseinschränkung. Was ist die wahrscheinlichste Diagnose?", "output": "Entzündung"},
    {"input": "Symptome: Plötzlicher starker Kopfschmerz, Übelkeit, Lichtscheu. Was ist die wahrscheinlichste Diagnose?", "output": "Migräne"},
    {"input": "Symptome: Husten mit Auswurf, Fieber, Leukozytose. Was ist die wahrscheinlichste Diagnose?", "output": "Bronchitis"},
    {"input": "Symptome: Sodbrennen, retrosternale Schmerzen, Reflux. Was ist die wahrscheinlichste Diagnose?", "output": "Refluxkrankheit"},
    {"input": "Symptome: Verstopfung, Bauchschmerzen, harter Stuhl. Was ist die wahrscheinlichste Diagnose?", "output": "Obstipation"},
    
    # Endokrinologie erweitert
    {"input": "Symptome: Durst, häufiges Wasserlassen, Gewichtsverlust. Was ist die wahrscheinlichste Diagnose?", "output": "Diabetes"},
    {"input": "Symptome: Kälteintoleranz, Gewichtszunahme, Haarausfall. Was ist die wahrscheinlichste Diagnose?", "output": "Hypothyreose"},
    {"input": "Symptome: Herzklopfen, Gewichtsverlust, Schwitzen. Was ist die wahrscheinlichste Diagnose?", "output": "Hyperthyreose"},
    {"input": "Symptome: Schwäche, Hypotonie, Hyperpigmentierung. Was ist die wahrscheinlichste Diagnose?", "output": "Addison-Krise"},
    {"input": "Symptome: Größenwachstum, vergrößerte Hände und Füße. Was ist die wahrscheinlichste Diagnose?", "output": "Akromegalie"},
    
    # Kardiologie erweitert
    {"input": "Symptome: Belastungsschmerzen in der Brust, EKG: ST-Senkungen. Was ist die wahrscheinlichste Diagnose?", "output": "Angina pectoris"},
    {"input": "Symptome: Herzrhythmusstörungen, Palpitationen, EKG: Vorhofflimmern. Was ist die wahrscheinlichste Diagnose?", "output": "Vorhofflimmern"},
    {"input": "Symptome: Beinschmerzen beim Gehen, ABI reduziert. Was ist die wahrscheinlichste Diagnose?", "output": "Arterielle Verschlusskrankheit"},
    {"input": "Symptome: Ödeme, Dyspnoe, Echokardiographie: reduzierte EF. Was ist die wahrscheinlichste Diagnose?", "output": "Herzinsuffizienz"},
    {"input": "Symptome: Thoraxschmerz, Perikardreiben, EKG: konkave ST-Hebungen. Was ist die wahrscheinlichste Diagnose?", "output": "Perikarditis"},
    
    # Gastroenterologie erweitert
    {"input": "Symptome: Oberbauchschmerzen, fettiges Essen, Sonographie: Gallensteine. Was ist die wahrscheinlichste Diagnose?", "output": "Cholelithiasis"},
    {"input": "Symptome: Sodbrennen, Völlegefühl, Helicobacter pylori positiv. Was ist die wahrscheinlichste Diagnose?", "output": "Gastritis"},
    {"input": "Symptome: Wechselnde Stuhlgewohnheiten, Blähungen, Bauchschmerzen. Was ist die wahrscheinlichste Diagnose?", "output": "Reizdarmsyndrom"},
    {"input": "Symptome: Schluckbeschwerden, retrosternaler Schmerz, Endoskopie: Ösophagitis. Was ist die wahrscheinlichste Diagnose?", "output": "Ösophagitis"},
    {"input": "Symptome: Übelkeit, Erbrechen, Epigastrium druckschmerzhaft. Was ist die wahrscheinlichste Diagnose?", "output": "Gastroenteritis"},
    
    # Pneumologie erweitert
    {"input": "Symptome: Trockener Husten, Belastungsdyspnoe, Röntgen: Wabenlunge. Was ist die wahrscheinlichste Diagnose?", "output": "Lungenfibrose"},
    {"input": "Symptome: Anfallsweise Dyspnoe, Giemen, Spirometrie: Obstruktion. Was ist die wahrscheinlichste Diagnose?", "output": "Asthma"},
    {"input": "Symptome: Produktiver Husten, Auswurf, Röntgen: Infiltrat. Was ist die wahrscheinlichste Diagnose?", "output": "Pneumonie"},
    {"input": "Symptome: Bluthusten, Gewichtsverlust, Röntgen: Raumforderung. Was ist die wahrscheinlichste Diagnose?", "output": "Bronchialkarzinom"},
    {"input": "Symptome: Akute Dyspnoe, Zyanose, Sauerstoffsättigung niedrig. Was ist die wahrscheinlichste Diagnose?", "output": "Respiratorische Insuffizienz"},
    
    # Neurologie erweitert
    {"input": "Symptome: Kopfschmerzen, Sehstörungen, Papillenödem. Was ist die wahrscheinlichste Diagnose?", "output": "Erhöhter Hirndruck"},
    {"input": "Symptome: Muskelkrämpfe, Faszikulationen, Atrophien. Was ist die wahrscheinlichste Diagnose?", "output": "ALS"},
    {"input": "Symptome: Vergesslichkeit, Orientierungsstörungen, MMST reduziert. Was ist die wahrscheinlichste Diagnose?", "output": "Demenz"},
    {"input": "Symptome: Schwindel, Gleichgewichtsstörungen, Nystagmus. Was ist die wahrscheinlichste Diagnose?", "output": "Vestibulopathie"},
    {"input": "Symptome: Lähmungserscheinungen, Sensibilitätsstörungen, MRT: Myelon. Was ist die wahrscheinlichste Diagnose?", "output": "Myelopathie"},
    
    # Infektiologie erweitert
    {"input": "Symptome: Fieber, Abgeschlagenheit, Lymphadenopathie generalisiert. Was ist die wahrscheinlichste Diagnose?", "output": "Viraler Infekt"},
    {"input": "Symptome: Harnwegssymptome, Fieber, Urinkultur: E. coli. Was ist die wahrscheinlichste Diagnose?", "output": "Harnwegsinfekt"},
    {"input": "Symptome: Hautausschlag, Fieber, Zeckenstich anamnestisch. Was ist die wahrscheinlichste Diagnose?", "output": "Borreliose"},
    {"input": "Symptome: Gelenkschmerzen, Fieber, ASL-Titer erhöht. Was ist die wahrscheinlichste Diagnose?", "output": "Rheumatisches Fieber"},
    {"input": "Symptome: Wunde infiziert, Rötung, Schwellung, Eiter. Was ist die wahrscheinlichste Diagnose?", "output": "Wundinfektion"},
    
    # Hämatologie erweitert
    {"input": "Symptome: Müdigkeit, Schwäche, Eisenmangel im Labor. Was ist die wahrscheinlichste Diagnose?", "output": "Eisenmangelanämie"},
    {"input": "Symptome: Blutergüsse, verlängerte Blutung, Quick erniedrigt. Was ist die wahrscheinlichste Diagnose?", "output": "Gerinnungsstörung"},
    {"input": "Symptome: Vergrößerte Lymphknoten, Nachtschweiß, Gewichtsverlust. Was ist die wahrscheinlichste Diagnose?", "output": "Lymphom"},
    {"input": "Symptome: Blässe, Herzrasen, Hämoglobin stark erniedrigt. Was ist die wahrscheinlichste Diagnose?", "output": "Schwere Anämie"},
    {"input": "Symptome: Milzvergrößerung, Vollblutbild: Zytopenie. Was ist die wahrscheinlichste Diagnose?", "output": "Hypersplenismus"},
    
    # Rheumatologie erweitert
    {"input": "Symptome: Gelenkschmerzen, Steifigkeit, Röntgen: Gelenkspaltverschmälerung. Was ist die wahrscheinlichste Diagnose?", "output": "Arthrose"},
    {"input": "Symptome: Muskelschmerzen, Schwäche, CK erhöht. Was ist die wahrscheinlichste Diagnose?", "output": "Myositis"},
    {"input": "Symptome: Fingergelenke geschwollen, symmetrisch betroffen. Was ist die wahrscheinlichste Diagnose?", "output": "Rheumatoide Arthritis"},
    {"input": "Symptome: Wirbelsäulenschmerzen, Bewegungseinschränkung, Entzündungszeichen. Was ist die wahrscheinlichste Diagnose?", "output": "Spondylarthritis"},
    {"input": "Symptome: Hautveränderungen, Gelenkschwellungen, ANA positiv. Was ist die wahrscheinlichste Diagnose?", "output": "Kollagenose"},
    
    # Nephrologie erweitert
    {"input": "Symptome: Rückenschmerzen, Fieber, Klopfschmerz Nierenlager. Was ist die wahrscheinlichste Diagnose?", "output": "Pyelonephritis"},
    {"input": "Symptome: Schaumiger Urin, Ödeme, Proteinurie massiv. Was ist die wahrscheinlichste Diagnose?", "output": "Nephrotisches Syndrom"},
    {"input": "Symptome: Hypertonie, Hämaturie, Proteinurie. Was ist die wahrscheinlichste Diagnose?", "output": "Glomerulonephritis"},
    {"input": "Symptome: Polyurie, Polydipsie, Durst, ADH-Mangel. Was ist die wahrscheinlichste Diagnose?", "output": "Diabetes insipidus"},
    {"input": "Symptome: Elektrolytstörungen, Azidose, Urämie. Was ist die wahrscheinlichste Diagnose?", "output": "Nierenversagen"},
    
    # Weitere Notfälle
    {"input": "Symptome: Atemnot, Thoraxschmerz, Tachykardie, Wells-Score hoch. Was ist die wahrscheinlichste Diagnose?", "output": "Lungenembolie"},
    {"input": "Symptome: Akuter Bauch, Abwehrspannung, Leukozytose. Was ist die wahrscheinlichste Diagnose?", "output": "Akutes Abdomen"},
    {"input": "Symptome: Schock, Hypotonie, Tachykardie, Fieber. Was ist die wahrscheinlichste Diagnose?", "output": "Septischer Schock"},
    {"input": "Symptome: Bewusstseinstrübung, Verwirrtheit, Exsikkose. Was ist die wahrscheinlichste Diagnose?", "output": "Dehydratation"},
    {"input": "Symptome: Krampfanfall, Bewusstlosigkeit, postiktal. Was ist die wahrscheinlichste Diagnose?", "output": "Epileptischer Anfall"},
    
    # Gynäkologie/Urologie erweitert
    {"input": "Symptome: Brennen beim Wasserlassen, Harndrang, Urin trüb. Was ist die wahrscheinlichste Diagnose?", "output": "Zystitis"},
    {"input": "Symptome: Hodenschmerzen, Schwellung, Fieber. Was ist die wahrscheinlichste Diagnose?", "output": "Epididymitis"},
    {"input": "Symptome: Menstruationsstörungen, Hitzewallungen, FSH erhöht. Was ist die wahrscheinlichste Diagnose?", "output": "Menopause"},
    {"input": "Symptome: Unterleibsschmerzen, Fieber, Fluor. Was ist die wahrscheinlichste Diagnose?", "output": "Adnexitis"},
    {"input": "Symptome: Ausbleiben der Menstruation, Übelkeit, HCG positiv. Was ist die wahrscheinlichste Diagnose?", "output": "Schwangerschaft"},
    
    # Dermatologie erweitert
    {"input": "Symptome: Juckreiz, Hautrötung, Bläschenbildung. Was ist die wahrscheinlichste Diagnose?", "output": "Ekzem"},
    {"input": "Symptome: Schuppen, Juckreiz, seborrhoische Verteilung. Was ist die wahrscheinlichste Diagnose?", "output": "Seborrhoisches Ekzem"},
    {"input": "Symptome: Warzen, HPV-Nachweis, genitale Lokalisation. Was ist die wahrscheinlichste Diagnose?", "output": "Condylomata acuminata"},
    {"input": "Symptome: Pigmentfleck verändert, asymmetrisch, mehrfarbig. Was ist die wahrscheinlichste Diagnose?", "output": "Melanom-Verdacht"},
    {"input": "Symptome: Hautknötchen, Juckreiz, nach Insektenstich. Was ist die wahrscheinlichste Diagnose?", "output": "Urtikaria"}
]

data = pd.DataFrame(data)
print(f"GROSSE DATENBASIS: {len(data)} Beispiele für intensives Training")

tokenizer = T5Tokenizer.from_pretrained("t5-small")

# T5 TASK PREFIX FIX beibehalten
def tokenize_with_task_prefix(example):
    task_prefixed_input = f"medical diagnosis: {example['input']}"
    input_enc = tokenizer(task_prefixed_input, truncation=True, padding="max_length", max_length=128)
    output_enc = tokenizer(example["output"], truncation=True, padding="max_length", max_length=32)
    input_enc["labels"] = output_enc["input_ids"]
    return input_enc

print("✅ T5 Task Prefix wird beibehalten")

dataset = Dataset.from_pandas(data)
tokenized_dataset = dataset.map(tokenize_with_task_prefix)

# DataCollator Fix beibehalten
print("Before fix - Dataset features:", tokenized_dataset.features.keys())
tokenized_dataset = tokenized_dataset.remove_columns(["input", "output"])
print("After fix - Dataset features:", tokenized_dataset.features.keys())

model = T5ForConditionalGeneration.from_pretrained("t5-small")

# INTENSIVES TRAINING: Mehr Epochen für 100 Beispiele
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,  
    num_train_epochs=40,            
    logging_steps=10,
    save_strategy="no",
    learning_rate=3e-4,             
    warmup_steps=50,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

print("\n" + "="*70)
print("INTENSIVES TRAINING: 100 Beispiele, 40 Epochen")
print("Expected: Lernt Symptom → Diagnose Mapping")
print("="*70)

trainer.train()

# IMPROVED PREDICTION mit bewährten Parametern
def predict_improved(prompt):
    prefixed_prompt = f"medical diagnosis: {prompt}"
    inputs = tokenizer(prefixed_prompt, return_tensors="pt", padding=True, truncation=True)
    
    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=32,
        repetition_penalty=2.0,
        num_beams=4,
        early_stopping=True,
        eos_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# FINALE TESTS nach intensivem Training
test_cases = [
    "Symptome: Atemnot, Fieber, CRP 90, Röntgen: Infiltrat rechts. Was ist die wahrscheinlichste Diagnose?",
    "Symptome: Brustschmerz, Troponin hoch, EKG ST-Hebung. Was ist die wahrscheinlichste Diagnose?", 
    "Symptome: Polyurie, Polydipsie, BZ 320 mg/dl. Was ist die wahrscheinlichste Diagnose?",
    "Symptome: Tremor, Rigor, Bradykinesie. Was ist die wahrscheinlichste Diagnose?",
    "Symptome: Kopfschmerzen, Lichtscheu, Meningismus. Was ist die wahrscheinlichste Diagnose?"
]

print("\n" + "="*70)
print("FINALE TESTS - 100 Beispiele, 40 Epochen Training:")
print("="*70)

for i, test_prompt in enumerate(test_cases, 1):
    result = predict_improved(test_prompt)
    expected = ["Pneumonie", "Herzinfarkt", "Diabetes mellitus", "Morbus Parkinson", "Meningitis"][i-1]
    print(f"\nTest {i}:")
    print(f"Input: {test_prompt}")
    print(f"Generated: {result}")
    print(f"Expected: {expected}")
    print(f"Correct: {'✅' if expected.lower() in result.lower() else '❌'}")

print("\n" + "="*70)
print("INTENSIVES TRAINING ABGESCHLOSSEN!")
print("100 Beispiele aus 19 medizinischen Fachbereichen")
print("="*70)